需要有经典变分法、最优控制和 RL 的基础

这份笔记会带你从熟悉的确定性领域出发，一步步走进随机世界，力求每一步都扎实、优美。

---

# 随机最优控制学习笔记

---

## 第 1 章：从确定性到随机性——为什么需要新的数学？

### 1.1 确定性系统的优雅

你已经熟知，经典最优控制处理这样的系统：

$$
\dot{x}(t) = f(x(t), u(t), t), \quad x(0) = x_0
$$

目标是找到控制函数 $u(t)$，最小化代价泛函：

$$
J[u] = \int_0^T L(x, u, t)\,dt + \Phi(x(T))
$$

通过变分法，我们推导出：
- **庞特里亚金最大值原理**：引入伴随变量 $p(t)$，构造哈密顿量 $H = p \cdot f - L$，则最优控制满足 $\frac{\partial H}{\partial u} = 0$，伴随方程 $\dot{p} = -\frac{\partial H}{\partial x}$。
- **HJB 方程**：定义值函数 $V(x,t)$，它满足偏微分方程 $-\frac{\partial V}{\partial t} = \min_u \left[ L + \frac{\partial V}{\partial x} \cdot f \right]$。

一切都很美。状态轨迹是一条光滑曲线，优化对象是一个**函数**（开环控制 $u(t)$）或**反馈策略**（$u = \pi(x)$）。

### 1.2 当世界充满噪声

现在，我们踏入更真实的场景：系统受到随机扰动。

想象一架无人机在阵风中飞行，或者一个投资组合在波动市场中。系统的演化不再是确定性的，而是**随机过程**。我们用**随机微分方程** 来描述：

$$
dX_t = f(X_t, u_t, t)\,dt + \sigma(X_t, u_t, t)\,dW_t
$$

- $X_t$ 是状态，现在是一个随机过程。
- $W_t$ 是**布朗运动**，你对它已经很熟悉了：它是连续随机过程的基石，增量独立、服从 $\mathcal{N}(0, dt)$。
- $\sigma$ 是扩散项系数，决定了噪声的强度及其如何依赖于状态和控制。

**关键差异**：控制的目标不再是最小化一个确定的代价值，而是最小化**期望代价**：

$$
J[u] = \mathbb{E}\left[ \int_0^T L(X_t, u_t, t)\,dt + \Phi(X_T) \right]
$$

这里的期望 $\mathbb{E}[\cdot]$ 是针对所有可能的噪声轨迹取的。

### 1.3 挑战在哪里？

经典变分法面临两个新问题：
1.  **状态是随机的**：给控制一个微小扰动 $\delta u$，状态的变化 $\delta X_t$ 也是一个随机过程。我们无法再简单地对“一条轨迹”做微分，而必须处理一族轨迹。
2.  **噪声尺度是 $\sqrt{dt}$**：布朗运动的增量 $dW_t$ 的方差是 $dt$。当我们在随机环境下做泰勒展开时，$dW_t$ 的二阶项会产生 $dt$ 项，不能忽略。这就是**伊藤引理**的源头，也是随机微积分与普通微积分的分水岭。

**本章小结**：随机最优控制 = 在 SDE 约束下，优化期望代价。由于 $\sqrt{dt}$ 效应，我们需要一套新的变分工具——**随机变分法**。

---

## 第 2 章：随机变分法——核心推导

这一章是整个笔记的心脏。我们将用变分法推导**随机最大值原理**，你可以逐行对比它与经典推导的异同。

### 2.1 问题设定

系统动态：
$$
dX_t = f(X_t, u_t, t)\,dt + \sigma(X_t, u_t, t)\,dW_t, \quad X_0 \text{ 已知}
$$

期望代价：
$$
J[u] = \mathbb{E}\left[ \int_0^T L(X_t, u_t, t)\,dt + \Phi(X_T) \right]
$$

任务是：找到 $u_t$（允许是反馈 $u_t = \pi(X_t)$ 或开环 $u_t$），使 $J$ 最小。

### 2.2 第一步：施加控制变分

给候选控制 $u_t$ 一个微小“针状”或连续扰动，得到新控制：
$$
u_t^\epsilon = u_t + \delta u_t
$$
状态变为：
$$
X_t^\epsilon = X_t + \delta X_t
$$
代价变为 $J^\epsilon$。

### 2.3 第二步：状态变分 $\delta X_t$ 的 SDE

$X_t^\epsilon$ 满足：
$$
dX_t^\epsilon = f(X_t^\epsilon, u_t^\epsilon, t)\,dt + \sigma(X_t^\epsilon, u_t^\epsilon, t)\,dW_t
$$
原状态 $X_t$ 满足方程。将两者相减，并做一阶泰勒展开：
$$
\begin{aligned}
d(\delta X_t) &= dX_t^\epsilon - dX_t \\
&= [f(X_t^\epsilon, u_t^\epsilon, t) - f(X_t, u_t, t)] dt \\
&\quad + [\sigma(X_t^\epsilon, u_t^\epsilon, t) - \sigma(X_t, u_t, t)] dW_t \\
&\approx \left[ f_x \delta X_t + f_u \delta u_t \right] dt + \left[ \sigma_x \delta X_t + \sigma_u \delta u_t \right] dW_t
\end{aligned}
$$
（为简洁，偏导数 $f_x, f_u, \sigma_x, \sigma_u$ 均在原轨迹 $(X_t, u_t, t)$ 处取值。）

初始条件：$\delta X_0 = 0$。

**这是一个关于 $\delta X_t$ 的线性随机微分方程。**

### 2.4 第三步：代价的变分

代价变分：
$$
\begin{aligned}
\delta J &= \mathbb{E}\left[ \int_0^T \left( L(X_t^\epsilon, u_t^\epsilon, t) - L(X_t, u_t, t) \right) dt + \Phi(X_T^\epsilon) - \Phi(X_T) \right] \\
&\approx \mathbb{E}\left[ \int_0^T \left( L_x \delta X_t + L_u \delta u_t \right) dt + \Phi_x(X_T) \delta X_T \right]
\end{aligned}
$$
最优性必要条件：对所有 $\delta u$，$\delta J = 0$。

麻烦在于 $\delta X_t$ 通过 SDE 依赖于 $\delta u$。我们需要**伴随过程**来解耦。

### 2.5 第四步：引入伴随过程 $p_t$——消除 $\delta X$

假设存在一个**适应过程**（即随着时间展开才逐渐确定的随机过程）$p_t$，其动态待定。我们希望用它“分部积分”掉 $\delta X$。考虑 $p_t \cdot \delta X_t$ 的微分。根据**伊藤引理**：

$$
d(p_t \cdot \delta X_t) = p_t d(\delta X_t) + \delta X_t dp_t + dp_t d(\delta X_t)
$$

这一项 $dp_t d(\delta X_t)$ 正是 $\sqrt{dt}$ 效应的体现，在确定性微积分中它为零。我们假设 $p_t$ 遵循一般 SDE：
$$
dp_t = \alpha_t dt + \beta_t dW_t
$$
其中 $\alpha_t, \beta_t$ 待定。计算交叉变分：
$$
\begin{aligned}
dp_t d(\delta X_t) &= (\alpha_t dt + \beta_t dW_t) \cdot \left( [\cdots] dt + [\sigma_x \delta X_t + \sigma_u \delta u_t] dW_t \right) \\
&= \beta_t (\sigma_x \delta X_t + \sigma_u \delta u_t) dt
\end{aligned}
$$
这里利用了 $dt \cdot dt = 0, dt \cdot dW_t = 0$，而 $dW_t \cdot dW_t = dt$。

代入得：
$$
\begin{aligned}
d(p_t \delta X_t) &= p_t \left[ (f_x \delta X_t + f_u \delta u_t) dt + (\sigma_x \delta X_t + \sigma_u \delta u_t) dW_t \right] \\
&\quad + \delta X_t (\alpha_t dt + \beta_t dW_t) \\
&\quad + \beta_t (\sigma_x \delta X_t + \sigma_u \delta u_t) dt
\end{aligned}
$$

现在，积分并取期望：
$$
\mathbb{E}[p_T \delta X_T] - \underbrace{\mathbb{E}[p_0 \delta X_0]}_{=0} = \mathbb{E}\left[ \int_0^T d(p_t \delta X_t) \right]
$$
因为随机积分的期望为零（$dW_t$ 项的积分为零鞅），我们有：
$$
\mathbb{E}[p_T \delta X_T] = \mathbb{E}\left[ \int_0^T \left( p_t f_x \delta X_t + p_t f_u \delta u_t + \alpha_t \delta X_t + \beta_t \sigma_x \delta X_t + \beta_t \sigma_u \delta u_t \right) dt \right]
$$

### 2.6 第五步：拼图完成——伴随方程与最大值原理

我们的目标是利用 $\mathbb{E}[p_T \delta X_T]$ 来消去 $\delta J$ 中的 $\Phi_x \delta X_T$ 和 $L_x \delta X_t$。
设 $p_T = \Phi_x(X_T)$，则：
$$
\mathbb{E}[\Phi_x(X_T) \delta X_T] = \mathbb{E}[p_T \delta X_T]
$$

将此代入 $\delta J$ 表达式：
$$
\begin{aligned}
\delta J &= \mathbb{E}\left[ \int_0^T (L_x \delta X_t + L_u \delta u_t) dt + p_T \delta X_T \right] \\
&= \mathbb{E}\left[ \int_0^T (L_x \delta X_t + L_u \delta u_t) dt + \int_0^T d(p_t \delta X_t) \right] \\
&= \mathbb{E}\left[ \int_0^T \left( L_x \delta X_t + L_u \delta u_t + p_t f_x \delta X_t + p_t f_u \delta u_t + \alpha_t \delta X_t + \beta_t \sigma_x \delta X_t + \beta_t \sigma_u \delta u_t \right) dt \right]
\end{aligned}
$$

合并 $\delta X_t$ 和 $\delta u_t$ 的项：
$$
\delta J = \mathbb{E}\left[ \int_0^T \left( (L_x + p_t f_x + \alpha_t + \beta_t \sigma_x) \delta X_t + (L_u + p_t f_u + \beta_t \sigma_u) \delta u_t \right) dt \right]
$$

**现在，施展变分法的魔法**：我们要让 $\delta J = 0$ 对任意 $\delta u$ 成立。首先，消除 $\delta X_t$ 的影响。我们无法直接控制 $\delta X_t$，但我们可以**选择** $\alpha_t$ 使得 $\delta X_t$ 的系数为零：
$$
\alpha_t = -L_x - p_t f_x - \beta_t \sigma_x
$$
这样，$\delta J$ 简化为：
$$
\delta J = \mathbb{E}\left[ \int_0^T \left( L_u + p_t f_u + \beta_t \sigma_u \right) \delta u_t dt \right]
$$

令其对任意 $\delta u_t$ 为零，得到最优控制的必要条件：
$$
L_u + p_t f_u + \beta_t \sigma_u = 0, \quad \text{a.e. } t, \text{ a.s.}
$$

### 2.7 随机最大值原理的完整表述

1.  **定义随机哈密顿量** $\mathcal{H}$：
    $$
    \boxed{\mathcal{H}(x,u,p,\beta,t) = p^\top f(x,u,t) + \text{Tr}(\beta^\top \sigma(x,u,t)) - L(x,u,t)}
    $$
    （在多维情况下，$p$ 是行向量，$\beta$ 是矩阵，$\text{Tr}$ 为迹。）

2.  **正向 SDE**：状态方程
    $$
    dX_t = \frac{\partial \mathcal{H}}{\partial p}(X_t, u_t, t)\,dt + \sigma(X_t, u_t, t)\,dW_t
    $$

3.  **倒向 SDE (BSDE)**：伴随方程
    $$
    \boxed{dp_t = -\frac{\partial \mathcal{H}}{\partial x}(X_t, u_t, p_t, \beta_t, t)\,dt + \beta_t dW_t}
    $$
    终端条件：$p_T = -\frac{\partial \Phi}{\partial x}(X_T)$（符号取决于 $\mathcal{H}$ 定义，此处与之前推导自洽）。

4.  **最大值原理**：最优控制 $u^*$ 使得
    $$
    \boxed{\mathcal{H}(X_t, u^*, p_t, \beta_t, t) = \max_{u \in U} \mathcal{H}(X_t, u, p_t, \beta_t, t)}
    $$
    如果 $u$ 无约束，则有 $\frac{\partial \mathcal{H}}{\partial u} = 0$。

**你的知识锚点**：对比经典的最大值原理，这里唯一的新元素是**伴随过程的扩散项 $\beta_t$** 和哈密顿量中多出的 **迹项** $\text{Tr}(\beta^\top \sigma)$。$\beta_t$ 可直观理解为“值函数对噪声的敏感度”。

---

## 第 3 章：从另一个角度看——随机动态规划与 HJB 方程

变分法给出的是“必要条件”。动态规划给出的是“充分条件”，并且直接得到反馈策略。

### 3.1 定义值函数

$$
V(x,t) = \min_{u_{[t,T]}} \mathbb{E}\left[ \int_t^T L(X_s, u_s, s)\,ds + \Phi(X_T) \mid X_t = x \right]
$$

### 3.2 推导 HJB 方程

根据动态规划原理，将区间分为 $[t, t+dt]$ 和 $[t+dt, T]$：
$$
V(x,t) = \min_{u} \mathbb{E}\left[ \int_t^{t+dt} L ds + V(X_{t+dt}, t+dt) \mid X_t = x \right]
$$
对 $V(X_{t+dt}, t+dt)$ 应用**伊藤引理**：
$$
\begin{aligned}
dV(X_t, t) &= \left( V_t + V_x^\top f + \frac{1}{2}\text{Tr}(\sigma \sigma^\top V_{xx}) \right) dt + V_x^\top \sigma dW_t
\end{aligned}
$$
代入并取期望，$dW_t$ 项消失，除以 $dt$，得到：

$$
\boxed{-V_t(x,t) = \min_{u} \left[ L(x,u,t) + V_x^\top f(x,u,t) + \frac{1}{2}\text{Tr}(\sigma(x,u,t) \sigma(x,u,t)^\top V_{xx}(x,t)) \right]}
$$

终端条件：$V(x,T) = \Phi(x)$。

**你的知识锚点**：对比确定性的 HJB 方程，这里多出了二阶导数项 $\frac{1}{2}\text{Tr}(\sigma \sigma^\top V_{xx})$。这同样是布朗运动 $\sqrt{dt}$ 尺度的直接数学后果。

### 3.3 两种方法的联系

如果值函数 $V$ 足够光滑，那么 **HJB 方程的解与随机最大值原理完全等价**。
它们之间的联系是：
$$
p_t = V_x(X_t, t), \quad \beta_t = V_{xx}(X_t, t) \sigma(X_t, u_t, t)
$$
将这两个关系代入随机最大值原理，就会得到 HJB 方程。所以，**它们是一个硬币的两面。**

---

## 第 4 章：线性二次型 (LQ) 问题——手算实战

当系统线性、代价二次型时，问题可完全解析求解。这是随机 LQR。

**系统**：
$$
dX_t = (A X_t + B u_t) dt + \Sigma dW_t
$$
**代价**：
$$
J = \mathbb{E}\left[ \int_0^T (X_t^\top Q X_t + u_t^\top R u_t) dt + X_T^\top M X_T \right]
$$
**解**：
1.  **值函数**设为二次型：$V(x,t) = x^\top P(t) x + c(t)$。
2.  **HJB 方程**代入，得到 $c(t)$ 的 ODE，和著名的**微分 Riccati 方程**：
    $$
    \boxed{-\dot{P}(t) = PA + A^\top P - P B R^{-1} B^\top P + Q, \quad P(T) = M}
    $$
3.  **最优控制**为线性状态反馈：
    $$
    \boxed{u^*(t, x) = -R^{-1} B^\top P(t) x}
    $$

**你的知识锚点**：这个结果与确定性 LQR 在形式上**完全一致**！噪声 $\Sigma$ 只进入了值函数的偏移项 $c(t)$ 中，不影响最优控制律。这就是**分离原理**的一种体现。

---

## 第 5 章：与强化学习的终极统一

现在，所有线索汇聚一堂。

- **RL 中的 MDP** 是随机最优控制在离散时空中的版本。
- **随机最优控制（连续时间）** 是 MDP 在时间无限细极限下的形式。
- **DQN/PPO/SAC** 等算法，是在模型（$f, \sigma$）未知时，用采样和神经网络来**近似求解 HJB 方程或随机最大值原理**。
- **最大熵 RL** 对偶于**路径积分控制**，后者是随机最优控制的一个分支，它将控制问题转化为纯粹的概率推断。其核心思想是：用 Girsanov 定理，选择一种控制，相当于在 Wiener 测度上施加一个漂移，从而变换整条路径的分布。

你走过的完整地图：
```
经典最优控制 (变分法/HJB, 确定性)
    │  + 布朗运动 $\sqrt{dt}$
    ▼
随机最优控制 (随机变分法/HJB, SDE)
    │  + 模型未知, 采样近似
    ▼
强化学习 (MDP, DQN/PPO/SAC)
```
